# RHINO Spectrometer Comparison v7
## RFSoC 4x2 | QICK 0.2.388 | PYNQ 3.0.1
### Mbatshi Jerry Junior Mbulawa | Jodrell Bank Observatory
### Supervised by Dr. Phil Bull

---

**What this notebook does:**
Captures raw ADC data from the RFSoC 4x2 board and processes it through two software-defined spectrometers — an FFT spectrometer and a Polyphase Filter Bank (PFB) spectrometer — then saves figures and data arrays for thesis analysis.

**Science band:** 60–85 MHz (EoR global signal)

**Key fixes in v7 vs v6:**
- PFB normalisation corrected: signal normalisation step removed, both spectrometers now give correct sigma^2 per bin
- Science band corrected to 60–85 MHz
- EDGES marker removed, band shading used instead
- Configurable N_TAPS (4 or 8)
- Frame cadence target: 10 seconds per waterfall frame
- Single longer waterfall run (360 frames default)
- Three waterfall views: full band, quiet sub-band, RFI sub-band

**Run all cells top to bottom. Do not skip cells.**

In [1]:
# ================================================================
# CELL 1 — Imports
# All Python libraries needed for the notebook.
# Run this first every time — nothing works without it.
# ================================================================
import numpy as np
import matplotlib
matplotlib.use('Agg')   # non-interactive backend — safe on RFSoC
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.gridspec import GridSpec
import os, sys, time, datetime, warnings
warnings.filterwarnings('ignore')

print('NumPy    :', np.__version__)
print('Matplotlib:', matplotlib.__version__)
print('Python   :', sys.version.split()[0])
print('PASS Cell 1 — imports OK')

NumPy    : 1.21.5
Matplotlib: 3.5.1
Python   : 3.10.4
PASS Cell 1 — imports OK


In [14]:
# ================================================================
# CELL 2 — Configuration
# All observation parameters in one place.
# Change OBSERVATION_MODE and N_TAPS here — nothing else needs
# to be changed for a standard observation.
# ================================================================

# ── Observation mode ─────────────────────────────────────────────
# 'desk' : lab bench, whip antenna, no LNA, low attenuation
# 'jbo'  : Jodrell Bank Observatory, discone antenna, ZKL-2+ LNA
OBSERVATION_MODE = 'jbo'

# ── PFB taps ─────────────────────────────────────────────────────
# 4 taps : standard, lower computational cost
# 8 taps : better sidelobe suppression, 2x more samples needed
N_TAPS = 4

# ── Science band ─────────────────────────────────────────────────
RHINO_LO_MHZ = 60.0    # RHINO science band lower edge
RHINO_HI_MHZ = 85.0    # RHINO science band upper edge

# Sub-band definitions for zoomed waterfall plots
SUBBAND_QUIET_LO = 65.0   # quietest sub-band (avoids 82 MHz spike)
SUBBAND_QUIET_HI = 78.0
SUBBAND_RFI_LO   = 60.0   # most RFI-active sub-band
SUBBAND_RFI_HI   = 70.0

# ── Hardware constants ────────────────────────────────────────────
ADC_CH              = 0          # ADC channel (ADC_D, tile 0 block 0)
SAMPLES_PER_TRANSFER = 1024     # DDR4 transfer size in samples
ADC_FULL_SCALE      = 8191      # 14-bit ADC maximum value
CLIP_THRESHOLD      = 0.95      # flag if |sample| > 95% of full scale

# ── FFT lengths ───────────────────────────────────────────────────
N_FFT_COARSE = 16384      # coarse: 270 kHz/bin
N_FFT_HIRES  = 1048576    # hi-res: 4.219 kHz/bin
N_BLOCK_COARSE = N_FFT_COARSE * N_TAPS   # samples needed for PFB
N_BLOCK_HIRES  = N_FFT_HIRES             # samples for hi-res FFT

# ── Waterfall settings ────────────────────────────────────────────
WF_FRAMES         = 60    # number of waterfall frames
                           # 360 frames x ~10s/frame = ~1 hour
WF_FRAME_TARGET_S = 10.0  # target wall-clock seconds per frame

# ── Mode-specific settings ────────────────────────────────────────
if OBSERVATION_MODE == 'desk':
    ANTENNA_LABEL      = 'Whip antenna (desk)'
    ADC_ATTENUATION_DB = 0
    USE_COBRA_DDR4     = False
elif OBSERVATION_MODE == 'jbo':
    ANTENNA_LABEL      = '50Ohm Load + ZKL-2+ LNA (JBO)'
    ADC_ATTENUATION_DB = 20
    USE_COBRA_DDR4     = False
else:
    raise ValueError('OBSERVATION_MODE must be desk or jbo')

# ── Save path ─────────────────────────────────────────────────────
SAVE_PATH = '/home/xilinx/jupyter_notebooks/RHINO/data/Load_LNA/'
os.makedirs(SAVE_PATH, exist_ok=True)

print('OBSERVATION_MODE  :', OBSERVATION_MODE)
print('ANTENNA_LABEL     :', ANTENNA_LABEL)
print('N_TAPS            :', N_TAPS)
print('Science band      : %.0f – %.0f MHz' % (RHINO_LO_MHZ, RHINO_HI_MHZ))
print('Quiet sub-band    : %.0f – %.0f MHz' % (SUBBAND_QUIET_LO, SUBBAND_QUIET_HI))
print('RFI sub-band      : %.0f – %.0f MHz' % (SUBBAND_RFI_LO, SUBBAND_RFI_HI))
print('WF_FRAMES         :', WF_FRAMES)
print('WF_FRAME_TARGET_S :', WF_FRAME_TARGET_S, 's')
print('Estimated duration: %.0f minutes' % (WF_FRAMES * WF_FRAME_TARGET_S / 60))
print('Save path         :', SAVE_PATH)
print('PASS Cell 2 — configuration OK')

OBSERVATION_MODE  : jbo
ANTENNA_LABEL     : 50Ohm Load + ZKL-2+ LNA (JBO)
N_TAPS            : 4
Science band      : 60 – 85 MHz
Quiet sub-band    : 65 – 78 MHz
RFI sub-band      : 60 – 70 MHz
WF_FRAMES         : 60
WF_FRAME_TARGET_S : 10.0 s
Estimated duration: 10 minutes
Save path         : /home/xilinx/jupyter_notebooks/RHINO/data/Load_LNA/
PASS Cell 2 — configuration OK


In [15]:
# ================================================================
# CELL 2b — Corrected NT values for QICK 0.2.388
# SAMPLES_PER_TRANSFER is 256 (not 1024) in this QICK version.
# NT = number of 256-sample transfers needed + FIFO latency buffer.
# These values are confirmed working from the original v6 notebook.
# ================================================================
SAMPLES_PER_TRANSFER = 256      # correct value for QICK 0.2.388

NT_COARSE = 68      # for N_FFT_COARSE = 16384  (16384/256 = 64, +4 FIFO)
NT_HIRES  = 4100    # for N_FFT_HIRES  = 1048576 (1048576/256 = 4096, +4 FIFO)
NT_WF     = 68      # same as NT_COARSE for waterfall frames

N_BLOCK_COARSE = N_FFT_COARSE * N_TAPS   # samples needed for PFB coarse
N_BLOCK_HIRES  = N_FFT_HIRES             # samples for hi-res FFT

print('SAMPLES_PER_TRANSFER : %d' % SAMPLES_PER_TRANSFER)
print('NT_COARSE            : %d  (covers %d samples)' % (NT_COARSE, NT_COARSE*SAMPLES_PER_TRANSFER))
print('NT_HIRES             : %d  (covers %d samples)' % (NT_HIRES, NT_HIRES*SAMPLES_PER_TRANSFER))
print('N_FFT_COARSE needed  : %d' % N_FFT_COARSE)
print('N_FFT_HIRES needed   : %d' % N_FFT_HIRES)
print('Coarse OK:', NT_COARSE*SAMPLES_PER_TRANSFER >= N_FFT_COARSE)
print('Hires  OK:', NT_HIRES*SAMPLES_PER_TRANSFER  >= N_FFT_HIRES)
print('PASS Cell 2b')

SAMPLES_PER_TRANSFER : 256
NT_COARSE            : 68  (covers 17408 samples)
NT_HIRES             : 4100  (covers 1049600 samples)
N_FFT_COARSE needed  : 16384
N_FFT_HIRES needed   : 1048576
Coarse OK: True
Hires  OK: True
PASS Cell 2b


In [16]:
# Cell 2c — Corrected NT values accounting for PFB sample requirement
# PFB needs N_TAPS * N_FFT_COARSE samples = 4 * 16384 = 65536
# FFT only needs N_FFT_COARSE = 16384 samples
# We capture enough for the PFB in all cases (FFT just uses the first N samples)

# Work out actual samples per transfer from what we received
# 68 transfers gave ~16607 samples, so ~244 samples per transfer
_ACTUAL_SPT = 16607 // 68   # estimate from observed data
print('Estimated samples per transfer:', _ACTUAL_SPT)

# NT values — add generous buffer for FIFO latency
NT_COARSE = (N_FFT_COARSE * N_TAPS // _ACTUAL_SPT) + 20   # enough for PFB
NT_HIRES  = (N_FFT_HIRES          // _ACTUAL_SPT) + 20   # enough for hi-res FFT
NT_WF     = NT_COARSE   # waterfall uses same as coarse

print('NT_COARSE (for PFB): %d  -> ~%d samples (need %d)' % (
      NT_COARSE, NT_COARSE * _ACTUAL_SPT, N_FFT_COARSE * N_TAPS))
print('NT_HIRES            : %d  -> ~%d samples (need %d)' % (
      NT_HIRES, NT_HIRES * _ACTUAL_SPT, N_FFT_HIRES))
print('PASS Cell 2c')

Estimated samples per transfer: 244
NT_COARSE (for PFB): 288  -> ~70272 samples (need 65536)
NT_HIRES            : 4317  -> ~1053348 samples (need 1048576)
PASS Cell 2c


In [17]:
# ================================================================
# CELL 3 — Board setup
# Loads the QICK firmware onto the RFSoC FPGA and reads the
# hardware configuration (sample rate, channel mapping, etc.)
# This cell communicates with the physical board.
# ================================================================
from qick import QickSoc, QickConfig
from qick.averager_program import AveragerProgram

soc = QickSoc()

# Read ADC sample rate from board
try:
    FS_MHZ = soc.readouts[ADC_CH].fs
except AttributeError:
    try:
        FS_MHZ = soc._cfg['readouts'][ADC_CH]['fs']
    except Exception:
        FS_MHZ = 4423.680   # confirmed value for RFSoC 4x2
        print('WARN: using hardcoded FS_MHZ =', FS_MHZ)

print('ADC sample rate   : %.3f Msps' % FS_MHZ)
print('Nyquist bandwidth : 0 – %.2f MHz' % (FS_MHZ / 2))
print('PASS Cell 3 — board setup OK')

ADC sample rate   : 4423.680 Msps
Nyquist bandwidth : 0 – 2211.84 MHz
PASS Cell 3 — board setup OK


In [18]:
# ================================================================
# CELL 4 — Frequency axes and science band masks
# Computes the frequency axis for each spectral resolution mode
# and creates boolean masks to extract the 60-85 MHz science band.
# ================================================================

# Coarse frequency axis (8193 bins, 0–2212 MHz)
freq_coarse = np.fft.rfftfreq(N_FFT_COARSE, d=1.0/FS_MHZ)   # MHz
DF_COARSE_KHZ = FS_MHZ * 1e3 / N_FFT_COARSE

# Hi-res frequency axis (524289 bins, 0–2212 MHz)
freq_hires  = np.fft.rfftfreq(N_FFT_HIRES,  d=1.0/FS_MHZ)   # MHz
DF_HIRES_KHZ  = FS_MHZ * 1e3 / N_FFT_HIRES

# Science band masks
def band_mask(freq, lo, hi):
    return (freq >= lo) & (freq <= hi)

mask_rhino_c  = band_mask(freq_coarse, RHINO_LO_MHZ, RHINO_HI_MHZ)
mask_rhino_h  = band_mask(freq_hires,  RHINO_LO_MHZ, RHINO_HI_MHZ)
mask_quiet_c  = band_mask(freq_coarse, SUBBAND_QUIET_LO, SUBBAND_QUIET_HI)
mask_rfi_c    = band_mask(freq_coarse, SUBBAND_RFI_LO,   SUBBAND_RFI_HI)

# Bin indices for science band (for waterfall slicing)
rhino_lo_c = int(np.searchsorted(freq_coarse, RHINO_LO_MHZ))
rhino_hi_c = int(np.searchsorted(freq_coarse, RHINO_HI_MHZ)) + 1
quiet_lo_c = int(np.searchsorted(freq_coarse, SUBBAND_QUIET_LO))
quiet_hi_c = int(np.searchsorted(freq_coarse, SUBBAND_QUIET_HI)) + 1
rfi_lo_c   = int(np.searchsorted(freq_coarse, SUBBAND_RFI_LO))
rfi_hi_c   = int(np.searchsorted(freq_coarse, SUBBAND_RFI_HI)) + 1

rhino_lo_h = int(np.searchsorted(freq_hires, RHINO_LO_MHZ))
rhino_hi_h = int(np.searchsorted(freq_hires, RHINO_HI_MHZ)) + 1

print('Coarse resolution : %.1f kHz/bin' % DF_COARSE_KHZ)
print('Hires  resolution : %.3f kHz/bin' % DF_HIRES_KHZ)
print('Science band bins (coarse): %d' % mask_rhino_c.sum())
print('Science band bins (hires) : %d' % mask_rhino_h.sum())
print('Quiet sub-band bins       : %d  (%.0f–%.0f MHz)' % (
      mask_quiet_c.sum(), SUBBAND_QUIET_LO, SUBBAND_QUIET_HI))
print('PASS Cell 4 — frequency axes OK')

Coarse resolution : 270.0 kHz/bin
Hires  resolution : 4.219 kHz/bin
Science band bins (coarse): 92
Science band bins (hires) : 5926
Quiet sub-band bins       : 48  (65–78 MHz)
PASS Cell 4 — frequency axes OK


In [19]:
# ================================================================
# CELL 5 — Spectrometer kernels
# Builds the FFT window and PFB prototype filter.
#
# KEY FIX in v7:
# The PFB signal normalisation step has been REMOVED.
# Both spectrometers now give the correct sigma^2 per bin
# for a white noise input — verified to within 0.001 dB.
#
# Why this matters: if you have a signal with known power P,
# both spectrometers should report the same power P in each bin.
# The old version had a residual offset of ~3 dB between them.
# ================================================================

# ── FFT window (Hann) ─────────────────────────────────────────────
fft_window      = np.hanning(N_FFT_COARSE).astype(np.float64)
fft_window_norm = float(np.sum(fft_window**2))

# ── Hi-res FFT window ────────────────────────────────────────────
hires_window      = np.hanning(N_FFT_HIRES).astype(np.float32)
hires_window_norm = float(np.sum(hires_window.astype(np.float64)**2))

# ── PFB prototype filter (Hann-windowed sinc, K taps) ───────────
# IMPORTANT: NO signal normalisation step.
# The power norm sum(h**2) is sufficient for correct PSD.
pfb_len    = N_FFT_COARSE * N_TAPS
t_pfb      = np.arange(pfb_len, dtype=np.float64) - pfb_len // 2
pfb_coeffs = np.sinc(t_pfb / N_FFT_COARSE) * np.hanning(pfb_len)
# Note: no pfb_coeffs /= ... line here (removed in v7)
pfb_win_norm = float(np.sum(pfb_coeffs**2))

# ── Verification: both should give sigma^2 per bin ───────────────
_rng   = np.random.default_rng(99)
_sigma = 100.0
_noise = _rng.normal(0, _sigma, N_FFT_COARSE + pfb_len)
_fft_p = np.abs(np.fft.rfft(_noise[-N_FFT_COARSE:] * fft_window))**2 / fft_window_norm
_pfb_b = _noise[:pfb_len].reshape(N_TAPS, N_FFT_COARSE)
_pfb_f = np.sum(_pfb_b * pfb_coeffs.reshape(N_TAPS, N_FFT_COARSE), axis=0)
_pfb_p = np.abs(np.fft.rfft(_pfb_f))**2 / pfb_win_norm
_fft_mean = float(np.mean(_fft_p[1:-1]))
_pfb_mean = float(np.mean(_pfb_p[1:-1]))
_offset   = 10*np.log10(_pfb_mean/_fft_mean)
print('Normalisation verification (sigma=%.0f, target=%.0f)' % (_sigma, _sigma**2))
print('  FFT mean power : %.2f' % _fft_mean)
print('  PFB mean power : %.2f' % _pfb_mean)
print('  PFB-FFT offset : %+.4f dB  (target: < 0.1 dB)' % _offset)
print('  STATUS:', 'PASS' if abs(_offset) < 0.1 else 'WARN — check normalisation')
print()
print('FFT  : Hann N=%d | norm=%.2f' % (N_FFT_COARSE, fft_window_norm))
print('PFB  : Hann-sinc K=%d taps | len=%d | norm=%.2f' % (N_TAPS, pfb_len, pfb_win_norm))
print('Hires: Hann N=%d | norm=%.2f' % (N_FFT_HIRES, hires_window_norm))
print('PASS Cell 5 — kernels built')

Normalisation verification (sigma=100, target=10000)
  FFT mean power : 10321.48
  PFB mean power : 9808.55
  PFB-FFT offset : -0.2214 dB  (target: < 0.1 dB)
  STATUS: WARN — check normalisation

FFT  : Hann N=16384 | norm=6143.63
PFB  : Hann-sinc K=4 taps | len=65536 | norm=13045.21
Hires: Hann N=1048576 | norm=393215.63
PASS Cell 5 — kernels built


In [20]:
# Cell 5b — Extended normalisation verification (500 averages)
import numpy as np
rng_v = np.random.default_rng(42)
N_V   = 500
sigma_v = 100.0
fft_acc = np.zeros(N_FFT_COARSE//2 + 1)
pfb_acc = np.zeros(N_FFT_COARSE//2 + 1)
for _ in range(N_V):
    noise = rng_v.normal(0, sigma_v, pfb_len + N_FFT_COARSE).astype(np.float64)
    fft_acc += np.abs(np.fft.rfft(
        noise[-N_FFT_COARSE:] * fft_window))**2 / fft_window_norm
    block = noise[:pfb_len].reshape(N_TAPS, N_FFT_COARSE)
    wsum  = np.sum(block * pfb_coeffs.reshape(N_TAPS, N_FFT_COARSE), axis=0)
    pfb_acc += np.abs(np.fft.rfft(wsum))**2 / pfb_win_norm
fft_mean = float(np.mean((fft_acc/N_V)[1:-1]))
pfb_mean = float(np.mean((pfb_acc/N_V)[1:-1]))
print('N=%d averages | target sigma^2=%.0f' % (N_V, sigma_v**2))
print('FFT mean power : %.2f' % fft_mean)
print('PFB mean power : %.2f' % pfb_mean)
print('PFB-FFT offset : %+.4f dB' % (10*np.log10(pfb_mean/fft_mean)))
print('STATUS:', 'PASS' if abs(10*np.log10(pfb_mean/fft_mean)) < 0.1 else 'WARN')

N=500 averages | target sigma^2=10000
FFT mean power : 9997.83
PFB mean power : 9997.52
PFB-FFT offset : -0.0001 dB
STATUS: PASS


In [21]:
# ================================================================
# CELL 6 — Spectrometer functions
# Defines the two spectrometer functions used throughout the notebook.
# Both take raw ADC samples and return power in dB.
# ================================================================

def fft_spectrometer(samples, n_fft=N_FFT_COARSE):
    """Hann-windowed FFT power spectrum.
    Input : raw ADC samples (float array)
    Output: power spectrum in dB, shape (n_fft//2 + 1,)
    """
    s = np.abs(np.fft.rfft(
        samples[:n_fft].astype(np.float64) * fft_window, n=n_fft))**2 / fft_window_norm
    return (10 * np.log10(s + 1e-100)).astype(np.float32)

def pfb_spectrometer(samples, n_fft=N_FFT_COARSE, n_taps=N_TAPS):
    """Polyphase filter bank power spectrum (Hann-sinc, K taps).
    Input : raw ADC samples (float array, must have >= n_fft*n_taps samples)
    Output: power spectrum in dB, shape (n_fft//2 + 1,)
    """
    block = samples[:n_fft * n_taps].astype(np.float64)
    wsum  = np.sum(block.reshape(n_taps, n_fft) *
                   pfb_coeffs.reshape(n_taps, n_fft), axis=0)
    s     = np.abs(np.fft.rfft(wsum, n=n_fft))**2 / pfb_win_norm
    return (10 * np.log10(s + 1e-100)).astype(np.float32)

def fft_hires(samples):
    """Hann-windowed hi-res FFT power spectrum.
    Input : raw ADC samples (float array, >= N_FFT_HIRES samples)
    Output: power spectrum in dB, shape (N_FFT_HIRES//2 + 1,)
    """
    s = np.abs(np.fft.rfft(
        samples[:N_FFT_HIRES].astype(np.float64) *
        hires_window.astype(np.float64), n=N_FFT_HIRES))**2
    return (10 * np.log10(s / hires_window_norm + 1e-100)).astype(np.float32)

# Quick shape check
_t  = np.random.normal(0, 100, N_BLOCK_COARSE + N_FFT_COARSE)
_fo = fft_spectrometer(_t)
_po = pfb_spectrometer(_t)
ok  = len(_fo) == len(_po) == N_FFT_COARSE // 2 + 1
print('FFT output bins : %d' % len(_fo))
print('PFB output bins : %d' % len(_po))
print('PASS Cell 6 — spectrometer functions defined' if ok else 'FAIL shape mismatch')

FFT output bins : 8193
PFB output bins : 8193
PASS Cell 6 — spectrometer functions defined


In [22]:
# ================================================================
# CELL 7 — DDR4 capture function (fixed for QICK 0.2.388)
# ================================================================

def _ddr4_capture_raw(nt):
    """Capture nt DDR4 transfers. Returns float32 I-channel array."""
    if USE_COBRA_DDR4:
        _c = soc.get_cfg()
        soc.ddr4_buf.set_switch(_c['readouts'][ADC_CH]['avgbuf_fullpath'])
        soc.clear_ddr4()
        soc.ddr4_buf.arm(nt=nt)
        prog.acquire(soc, load_pulses=False, progress=False)
        raw = soc.ddr4_buf.get_mem(nt=nt)
        return (raw[:, 0] if raw.ndim == 2 else raw).astype(np.float32)
    else:
        soc.arm_ddr4(ch=ADC_CH, nt=nt)
        prog.acquire(soc, load_pulses=False, progress=False)
        # QICK 0.2.388 fix: get_ddr4() does not accept 'ch' keyword
        raw = soc.get_ddr4(nt=nt)
        return raw[:, 0].astype(np.float32)

print('DDR4 capture method:', 'CobraX' if USE_COBRA_DDR4 else 'Standard QICK 0.2.388')
print('PASS Cell 7 — DDR4 function defined')

DDR4 capture method: Standard QICK 0.2.388
PASS Cell 7 — DDR4 function defined


In [23]:
# ================================================================
# CELL 8 — QICK program setup (fixed for QICK 0.2.388, PYNQ ADC)
# ================================================================
from qick.averager_program import AveragerProgram

class RHINOReadoutProgram(AveragerProgram):
    def initialize(self):
        # PYNQ-configured ADC requires freq=0 in declaration
        self.declare_readout(ch=ADC_CH,
                             length=SAMPLES_PER_TRANSFER,
                             freq=0)
        self.synci(200)
    def body(self):
        # ddr4=True required for DDR4 buffer capture in QICK 0.2.388
        self.trigger(adcs=[ADC_CH],
                     ddr4=True,
                     adc_trig_offset=0,
                     t=0)
        self.synci(200)

cfg = {
    'adc_trig_offset': 0,
    'reps'           : 1,
    'rounds'         : 1,
    'soft_avgs'      : 1,
    'relax_delay'    : 0,
    'readout_length' : SAMPLES_PER_TRANSFER,
}
prog = RHINOReadoutProgram(soc, cfg)
print('PASS Cell 8 — QICK program ready')

PASS Cell 8 — QICK program ready


In [24]:
# ================================================================
# CELL 9 — Attenuation and clipping check
# Sets the ADC attenuation and verifies the signal is not clipping.
# Clipping means the signal is too strong and the ADC is saturating,
# which causes distortion. We check this before any science capture.
# ================================================================
try:
    soc.set_mixer_freq(ADC_CH, 0)
except Exception:
    pass

try:
    soc.set_attenuation(ADC_CH, ADC_ATTENUATION_DB)
    print('Attenuation set to %d dB' % ADC_ATTENUATION_DB)
except Exception as e:
    print('WARN: could not set attenuation:', e)

# Capture a short test burst and check for clipping
NT_TEST = NT_COARSE
test_raw = _ddr4_capture_raw(NT_TEST)
clip_pct = float(np.mean(np.abs(test_raw) > CLIP_THRESHOLD * ADC_FULL_SCALE)) * 100
peak_pct = float(np.max(np.abs(test_raw))) / ADC_FULL_SCALE * 100
rms_adu  = float(np.sqrt(np.mean(test_raw**2)))
atten_ok = clip_pct < 2.0

print('Peak / ADC_MAX : %.1f%%' % peak_pct)
print('RMS            : %.1f ADU' % rms_adu)
print('Clip fraction  : %.4f%%' % clip_pct)
print('STATUS         :', 'OK' if atten_ok else 'CLIPPING — increase attenuation')
if not atten_ok:
    print('ACTION: increase ADC_ATTENUATION_DB in Cell 2 and re-run from Cell 2')
final_atten = ADC_ATTENUATION_DB
print('PASS Cell 9 — clipping check complete')

WARN: could not set attenuation: Could not find IP or hierarchy set_attenuation in overlay
Peak / ADC_MAX : 0.3%
RMS            : 5.1 ADU
Clip fraction  : 0.0000%
STATUS         : OK
PASS Cell 9 — clipping check complete


In [25]:
# ================================================================
# CELL 10 — Wideband spectrum capture
# Captures a single spectrum covering the full 0–2212 MHz Nyquist
# bandwidth, and plots it with the RHINO science band highlighted.
# This gives a first look at the RFI environment before the
# detailed science band measurements.
# ================================================================
if not atten_ok:
    print('SKIP — fix clipping first (Cell 9)')
else:
    ts10 = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
    NT_WB = NT_COARSE

    # Average 10 spectra for a clean wideband view
    n_avg_wb = 10
    fft_wb = np.zeros(N_FFT_COARSE // 2 + 1, dtype=np.float64)
    pfb_wb = np.zeros(N_FFT_COARSE // 2 + 1, dtype=np.float64)
    for _ in range(n_avg_wb):
        raw = _ddr4_capture_raw(NT_WB)
        fft_wb += fft_spectrometer(raw).astype(np.float64)
        pfb_wb += pfb_spectrometer(raw).astype(np.float64)
    fft_wb /= n_avg_wb
    pfb_wb /= n_avg_wb

    # Save arrays
    np.save(SAVE_PATH + 'fft_coarse_%s.npy' % ts10, fft_wb.astype(np.float32))
    np.save(SAVE_PATH + 'pfb_coarse_%s.npy' % ts10, pfb_wb.astype(np.float32))
    np.save(SAVE_PATH + 'freq_coarse_%s.npy' % ts10, freq_coarse)

    # Plot 0–500 MHz
    wb_mask = freq_coarse <= 500.0
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(freq_coarse[wb_mask], fft_wb[wb_mask],
            color='steelblue', lw=0.7, alpha=0.85, label='FFT (Hann)')
    ax.plot(freq_coarse[wb_mask], pfb_wb[wb_mask],
            color='firebrick', lw=0.7, alpha=0.85, label='PFB (%d taps)' % N_TAPS)
    ax.axvspan(RHINO_LO_MHZ, RHINO_HI_MHZ, alpha=0.15, color='gold',
               label='RHINO band (%.0f–%.0f MHz)' % (RHINO_LO_MHZ, RHINO_HI_MHZ))
    ax.axvspan(87.5, 108.0, alpha=0.08, color='red', label='FM (87.5–108 MHz)')
    ax.set_xlabel('Frequency (MHz)')
    ax.set_ylabel('Power (dB, arb.)')
    ax.set_title('Wideband Spectrum 0–500 MHz | %s | %s' % (ts10, ANTENNA_LABEL))
    ax.legend(fontsize=8)
    ax.set_xlim(0, 500)
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(10))
    fig.tight_layout()
    out10 = SAVE_PATH + 'thesis_fig3_wideband_%s.png' % ts10
    fig.savefig(out10, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print('Saved:', out10)
    print('PASS Cell 10 — wideband spectrum complete')

Saved: /home/xilinx/jupyter_notebooks/RHINO/data/Load_LNA/thesis_fig3_wideband_20260520_002628.png
PASS Cell 10 — wideband spectrum complete


In [26]:
# ================================================================
# CELL 11 — RHINO band zoom (60–85 MHz)
# Zooms into the science band and plots FFT vs PFB side by side
# with a dual axis showing both frequency (MHz) and redshift (z).
# Also prints the spectral flatness metrics (std dev per spectrometer).
# ================================================================
if not atten_ok:
    print('SKIP — fix clipping first')
else:
    def freq_to_z(f): return 1420.405751786 / f - 1.0

    fft_rhino = fft_wb[mask_rhino_c]
    pfb_rhino = pfb_wb[mask_rhino_c]
    f_rhino   = freq_coarse[mask_rhino_c]

    fft_std = float(np.std(fft_rhino))
    pfb_std = float(np.std(pfb_rhino))

    fig, ax = plt.subplots(figsize=(10, 4))
    ax2 = ax.twiny()
    ax.plot(f_rhino, fft_rhino, color='steelblue', lw=1.2,
            label='FFT (Hann)  std=%.3f dB' % fft_std)
    ax.plot(f_rhino, pfb_rhino, color='firebrick', lw=1.2,
            label='PFB (%d taps)  std=%.3f dB' % (N_TAPS, pfb_std))
    ax.axvspan(SUBBAND_QUIET_LO, SUBBAND_QUIET_HI,
               alpha=0.08, color='green', label='Quiet sub-band')
    ax.set_xlabel('Frequency (MHz)')
    ax.set_ylabel('Power (dB, arb.)')
    ax.set_xlim(RHINO_LO_MHZ, RHINO_HI_MHZ)
    ax.set_title('RHINO Science Band 60–85 MHz | FFT vs PFB | %s' % ANTENNA_LABEL)
    ax.legend(fontsize=8)
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))
    f_ticks = np.linspace(RHINO_LO_MHZ, RHINO_HI_MHZ, 6)
    ax2.set_xlim(RHINO_LO_MHZ, RHINO_HI_MHZ)
    ax2.set_xticks(f_ticks)
    ax2.set_xticklabels(['z=%.1f' % freq_to_z(f) for f in f_ticks], fontsize=8)
    ax2.set_xlabel('Redshift  z = ν₂₁/ν − 1', fontsize=9)
    fig.tight_layout()
    out11 = SAVE_PATH + 'thesis_fig4_rhino_band_%s.png' % ts10
    fig.savefig(out11, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print('FFT std in RHINO band: %.4f dB' % fft_std)
    print('PFB std in RHINO band: %.4f dB' % pfb_std)
    print('PFB - FFT std        : %+.4f dB' % (pfb_std - fft_std))
    print('Saved:', out11)
    print('PASS Cell 11 — RHINO band zoom complete')

FFT std in RHINO band: 1.6566 dB
PFB std in RHINO band: 1.7436 dB
PFB - FFT std        : +0.0870 dB
Saved: /home/xilinx/jupyter_notebooks/RHINO/data/Load_LNA/thesis_fig4_rhino_band_20260520_002628.png
PASS Cell 11 — RHINO band zoom complete


In [28]:
# ================================================================
# CELL 12 — Waterfall capture (coarse: FFT vs PFB)
# The main science cell. Captures WF_FRAMES spectra and builds
# a time-frequency waterfall plot showing how the spectrum
# evolves over time. Target: 10 seconds per frame.
#
# Three views are saved:
# (a) Full RHINO band: 60–85 MHz
# (b) Quiet sub-band : 65–78 MHz (lower RFI)
# (c) RFI sub-band   : 60–70 MHz (highest RFI activity)
#
# Each frame: capture raw samples -> FFT spectrum -> PFB spectrum
# -> save one row to each waterfall array.
# ================================================================
if not atten_ok:
    print('SKIP — fix clipping first')
else:
    ts12 = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
    n_bins_rhino = rhino_hi_c - rhino_lo_c
    n_bins_quiet = quiet_hi_c - quiet_lo_c
    n_bins_rfi   = rfi_hi_c   - rfi_lo_c

    wf_fft_full  = np.zeros((WF_FRAMES, n_bins_rhino), dtype=np.float32)
    wf_pfb_full  = np.zeros((WF_FRAMES, n_bins_rhino), dtype=np.float32)
    wf_fft_quiet = np.zeros((WF_FRAMES, n_bins_quiet), dtype=np.float32)
    wf_pfb_quiet = np.zeros((WF_FRAMES, n_bins_quiet), dtype=np.float32)
    wf_fft_rfi   = np.zeros((WF_FRAMES, n_bins_rfi),   dtype=np.float32)
    wf_pfb_rfi   = np.zeros((WF_FRAMES, n_bins_rfi),   dtype=np.float32)
    wf_times     = np.zeros(WF_FRAMES)
    wf_rms       = np.zeros(WF_FRAMES)   # RMS per frame for stripe detection

    NT_WF = NT_COARSE
    t_wf  = time.time()
    n_ok  = 0

    print('[WF] Starting %d frames | %.0f kHz/bin | target %.0fs/frame' %
          (WF_FRAMES, DF_COARSE_KHZ, WF_FRAME_TARGET_S))
    print('[WF] Estimated duration: %.0f minutes' %
          (WF_FRAMES * WF_FRAME_TARGET_S / 60))

    for frame in range(WF_FRAMES):
        t_frame_start = time.time()
        try:
            raw  = _ddr4_capture_raw(NT_WF)
            clip = float(np.mean(np.abs(raw) > CLIP_THRESHOLD * ADC_FULL_SCALE)) * 100
            wf_rms[frame] = float(np.sqrt(np.mean(raw**2)))
            if clip <= 2.0:
                fft_row = fft_spectrometer(raw)
                pfb_row = pfb_spectrometer(raw)
                wf_fft_full[frame]  = fft_row[rhino_lo_c:rhino_hi_c]
                wf_pfb_full[frame]  = pfb_row[rhino_lo_c:rhino_hi_c]
                wf_fft_quiet[frame] = fft_row[quiet_lo_c:quiet_hi_c]
                wf_pfb_quiet[frame] = pfb_row[quiet_lo_c:quiet_hi_c]
                wf_fft_rfi[frame]   = fft_row[rfi_lo_c:rfi_hi_c]
                wf_pfb_rfi[frame]   = pfb_row[rfi_lo_c:rfi_hi_c]
                n_ok += 1
            wf_times[frame] = time.time() - t_wf
            if (frame + 1) % 30 == 0:
                elapsed = wf_times[frame]
                rate    = elapsed / (frame + 1)
                remain  = (WF_FRAMES - frame - 1) * rate / 60
                print('  Frame %d/%d | t=%.0fs | %.1fs/frame | ~%.0f min remaining' %
                      (frame + 1, WF_FRAMES, elapsed, rate, remain))
        except Exception as e:
            print('  Frame %d ERROR: %s' % (frame + 1, str(e)))

        # Enforce target frame cadence
        elapsed_frame = time.time() - t_frame_start
        if elapsed_frame < WF_FRAME_TARGET_S:
            time.sleep(WF_FRAME_TARGET_S - elapsed_frame)

    total_time = time.time() - t_wf
    print('[WF] Done: %d/%d OK | %.1f s total | %.1f s/frame actual' %
          (n_ok, WF_FRAMES, total_time, total_time / WF_FRAMES))

    # Save arrays
    np.save(SAVE_PATH + 'wf_fft_%s.npy' % ts12, wf_fft_full)
    np.save(SAVE_PATH + 'wf_pfb_%s.npy' % ts12, wf_pfb_full)
    np.save(SAVE_PATH + 'wf_fft_quiet_%s.npy' % ts12, wf_fft_quiet)
    np.save(SAVE_PATH + 'wf_pfb_quiet_%s.npy' % ts12, wf_pfb_quiet)
    np.save(SAVE_PATH + 'wf_fft_rfi_%s.npy'   % ts12, wf_fft_rfi)
    np.save(SAVE_PATH + 'wf_pfb_rfi_%s.npy'   % ts12, wf_pfb_rfi)
    np.save(SAVE_PATH + 'wf_times_%s.npy'     % ts12, wf_times)
    np.save(SAVE_PATH + 'wf_rms_%s.npy'       % ts12, wf_rms)
    print('PASS Cell 12 — waterfall data saved')

[WF] Starting 360 frames | 270 kHz/bin | target 10s/frame
[WF] Estimated duration: 60 minutes
  Frame 30/360 | t=290s | 9.7s/frame | ~53 min remaining
  Frame 60/360 | t=591s | 9.8s/frame | ~49 min remaining
  Frame 90/360 | t=891s | 9.9s/frame | ~45 min remaining
  Frame 120/360 | t=1191s | 9.9s/frame | ~40 min remaining
  Frame 150/360 | t=1491s | 9.9s/frame | ~35 min remaining
  Frame 180/360 | t=1792s | 10.0s/frame | ~30 min remaining
  Frame 210/360 | t=2092s | 10.0s/frame | ~25 min remaining
  Frame 240/360 | t=2392s | 10.0s/frame | ~20 min remaining
  Frame 270/360 | t=2693s | 10.0s/frame | ~15 min remaining
  Frame 300/360 | t=2993s | 10.0s/frame | ~10 min remaining
  Frame 330/360 | t=3293s | 10.0s/frame | ~5 min remaining
  Frame 360/360 | t=3593s | 10.0s/frame | ~0 min remaining
[WF] Done: 360/360 OK | 3603.4 s total | 10.0 s/frame actual
PASS Cell 12 — waterfall data saved


In [29]:
# ================================================================
# CELL 13 — Waterfall plots
# Produces the waterfall figures from the data captured in Cell 12.
# Three panels for each sub-band. The difference waterfall
# (FFT - PFB) shows where the two spectrometers disagree.
# Also identifies stripe frames (broadband power jumps).
# ================================================================
if not atten_ok:
    print('SKIP — fix clipping first')
else:
    def plot_waterfall(wf_fft, wf_pfb, times, freq_lo, freq_hi, label, fname):
        n_fr, n_bins = wf_fft.shape
        f_ax  = np.linspace(freq_lo, freq_hi, n_bins)
        wfdiff = wf_fft - wf_pfb
        combined = np.concatenate([wf_fft.ravel(), wf_pfb.ravel()])
        combined = combined[combined != 0]
        vmin = float(np.percentile(combined, 2))  if len(combined) else 30
        vmax = float(np.percentile(combined, 98)) if len(combined) else 70
        vlim = max(float(np.percentile(np.abs(wfdiff), 99)), 0.5)
        ext  = [f_ax[0], f_ax[-1], times[-1], 0]

        # Identify stripe frames: RMS > 2 sigma above median
        rms_med = np.median(wf_rms)
        rms_std = np.std(wf_rms)
        stripe_frames = np.where(wf_rms > rms_med + 2*rms_std)[0]

        fig = plt.figure(figsize=(18, 11))
        gs  = GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.32)

        for ax, wf, title, cmap, vm1, vm2 in [
            (fig.add_subplot(gs[0,0]), wf_fft,  'FFT (Hann)',         'viridis', vmin, vmax),
            (fig.add_subplot(gs[0,1]), wf_pfb,  'PFB (%d taps)'%N_TAPS,'viridis', vmin, vmax),
            (fig.add_subplot(gs[0,2]), wfdiff,  'FFT − PFB',         'RdBu_r', -vlim, vlim)]:
            im = ax.imshow(wf, aspect='auto', origin='upper',
                           extent=ext, cmap=cmap, vmin=vm1, vmax=vm2,
                           interpolation='nearest')
            # Mark stripe frames
            for sf in stripe_frames:
                if sf < len(times):
                    ax.axhline(times[sf], color='yellow', lw=0.5, alpha=0.5)
            ax.set_xlabel('Frequency (MHz)')
            ax.set_ylabel('Time (s)')
            ax.set_title(title, fontsize=10)
            plt.colorbar(im, ax=ax, label='dB')

        ax4 = fig.add_subplot(gs[1, :])
        mean_fft = np.mean(wf_fft, axis=0)
        mean_pfb = np.mean(wf_pfb, axis=0)
        ax4.plot(f_ax, mean_fft, color='steelblue', lw=1.2, label='FFT mean')
        ax4.plot(f_ax, mean_pfb, color='firebrick', lw=1.2, label='PFB mean')
        mean_diff_val = float(np.mean(np.abs(wfdiff)))
        ax4.set_xlabel('Frequency (MHz)')
        ax4.set_ylabel('Mean power (dB)')
        ax4.set_title('Time-averaged spectrum (%d frames) | mean|FFT−PFB|=%.4f dB | '
                      'Stripe frames: %d/%d' %
                      (n_fr, mean_diff_val, len(stripe_frames), n_fr))
        ax4.set_xlim(freq_lo, freq_hi)
        ax4.legend(fontsize=9)

        fig.suptitle('FFT vs PFB Waterfall | %s – %s MHz | %s | %s' %
                     (freq_lo, freq_hi, label, ANTENNA_LABEL), fontsize=10)
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print('  Saved:', fname)
        print('  Stripe frames detected: %d/%d' % (len(stripe_frames), n_fr))
        if len(stripe_frames) > 0:
            print('  Stripe frame indices:', stripe_frames[:10],
                  '...' if len(stripe_frames) > 10 else '')
        return mean_diff_val

    print('Generating waterfall figures...')
    wf_freq_full  = freq_coarse[rhino_lo_c:rhino_hi_c]
    wf_freq_quiet = freq_coarse[quiet_lo_c:quiet_hi_c]
    wf_freq_rfi   = freq_coarse[rfi_lo_c:rfi_hi_c]

    d1 = plot_waterfall(wf_fft_full, wf_pfb_full, wf_times,
                        RHINO_LO_MHZ, RHINO_HI_MHZ, 'Full band',
                        SAVE_PATH + 'thesis_fig5_waterfall_full_%s.png' % ts12)
    d2 = plot_waterfall(wf_fft_quiet, wf_pfb_quiet, wf_times,
                        SUBBAND_QUIET_LO, SUBBAND_QUIET_HI, 'Quiet sub-band',
                        SAVE_PATH + 'thesis_fig5_waterfall_quiet_%s.png' % ts12)
    d3 = plot_waterfall(wf_fft_rfi, wf_pfb_rfi, wf_times,
                        SUBBAND_RFI_LO, SUBBAND_RFI_HI, 'RFI sub-band',
                        SAVE_PATH + 'thesis_fig5_waterfall_rfi_%s.png' % ts12)
    print('PASS Cell 13 — waterfall plots complete')

Generating waterfall figures...
  Saved: /home/xilinx/jupyter_notebooks/RHINO/data/Yagi_LNA/thesis_fig5_waterfall_full_20260519_232717.png
  Stripe frames detected: 10/360
  Stripe frame indices: [ 12  15  26  39  79  86 156 239 304 325] 
  Saved: /home/xilinx/jupyter_notebooks/RHINO/data/Yagi_LNA/thesis_fig5_waterfall_quiet_20260519_232717.png
  Stripe frames detected: 10/360
  Stripe frame indices: [ 12  15  26  39  79  86 156 239 304 325] 
  Saved: /home/xilinx/jupyter_notebooks/RHINO/data/Yagi_LNA/thesis_fig5_waterfall_rfi_20260519_232717.png
  Stripe frames detected: 10/360
  Stripe frame indices: [ 12  15  26  39  79  86 156 239 304 325] 
PASS Cell 13 — waterfall plots complete


In [30]:
# ================================================================
# CELL 14 — Raw ADC time stream plot
# Plots the raw ADC voltage samples vs time — the signal as seen
# by the ADC before any spectral processing.
# Also compares a stripe frame vs a clean frame (if stripes exist).
# This is what Phil asked for to diagnose the waterfall stripes.
# ================================================================
if not atten_ok:
    print('SKIP — fix clipping first')
else:
    ts14   = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
    N_SHOW = 8192   # samples to show (~1.85 µs at 4423 Msps)
    dt_us  = 1.0 / FS_MHZ
    t_us   = np.arange(N_SHOW) * dt_us
    NT_TS  = (N_FFT_HIRES // SAMPLES_PER_TRANSFER) + 5

    print('Capturing time stream...')
    raw_ts = _ddr4_capture_raw(NT_TS)
    rms_ts = float(np.sqrt(np.mean(raw_ts**2)))
    peak_ts = float(np.max(np.abs(raw_ts)))

    n_rows = 2 if 'stripe_frames' in dir() and len(stripe_frames) > 0 else 1
    fig, axes = plt.subplots(n_rows, 1, figsize=(12, 4 * n_rows))
    if n_rows == 1:
        axes = [axes]

    axes[0].plot(t_us, raw_ts[:N_SHOW], lw=0.4, color='steelblue', alpha=0.85)
    axes[0].axhline( ADC_FULL_SCALE, color='red', ls='--', lw=1.0, alpha=0.7,
                    label='ADC clip level')
    axes[0].axhline(-ADC_FULL_SCALE, color='red', ls='--', lw=1.0, alpha=0.7)
    axes[0].set_xlabel('Time (µs)')
    axes[0].set_ylabel('ADC output (ADU)')
    axes[0].set_title('Raw ADC Time Stream | RMS=%.1f ADU | Peak=%.0f ADU | %s' %
                      (rms_ts, peak_ts, ANTENNA_LABEL))
    axes[0].set_ylim(-ADC_FULL_SCALE * 1.1, ADC_FULL_SCALE * 1.1)
    axes[0].legend(fontsize=8)

    if n_rows == 2:
        axes[1].plot(t_us, raw_ts[:N_SHOW], lw=0.4, color='orange', alpha=0.85,
                    label='Current capture (compare with stripe frame)')
        axes[1].set_xlabel('Time (µs)')
        axes[1].set_ylabel('ADC output (ADU)')
        axes[1].set_title('Note: to compare with a stripe frame, '
                          're-run this cell during an active stripe')
        axes[1].set_ylim(-ADC_FULL_SCALE * 1.1, ADC_FULL_SCALE * 1.1)

    fig.suptitle('Raw ADC Time Stream — RHINO | %s | First %d samples | '
                 'dt=%.4f µs/sample' % (ts14, N_SHOW, dt_us), fontsize=10)
    fig.tight_layout()
    out14 = SAVE_PATH + 'raw_timestream_%s.png' % ts14
    fig.savefig(out14, dpi=150, bbox_inches='tight')
    plt.close(fig)
    np.save(SAVE_PATH + 'raw_timestream_%s.npy' % ts14, raw_ts)
    print('RMS  : %.1f ADU' % rms_ts)
    print('Peak : %.0f ADU (ADC_MAX = %d)' % (peak_ts, ADC_FULL_SCALE))
    print('Saved:', out14)
    print('PASS Cell 14 — time stream complete')

Capturing time stream...
RMS  : 485.5 ADU
Peak : 8342 ADU (ADC_MAX = 8191)
Saved: /home/xilinx/jupyter_notebooks/RHINO/data/Yagi_LNA/raw_timestream_20260520_002846.png
PASS Cell 14 — time stream complete


In [31]:
# ================================================================
# CELL 15 — Hi-res spectrum capture
# Captures the full DDR4 buffer (N = 1,048,576 samples) and
# computes the hi-res FFT spectrum at 4.219 kHz/bin resolution.
# This meets Phil's <10 kHz spectral resolution requirement.
# ================================================================
if not atten_ok:
    print('SKIP — fix clipping first')
else:
    ts15 = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
    NT_HR = NT_HIRES

    print('Capturing hi-res spectrum...')
    raw_hr = _ddr4_capture_raw(NT_HR)
    spec_hr = fft_hires(raw_hr)

    np.save(SAVE_PATH + 'fft_hires_%s.npy' % ts15, spec_hr)
    np.save(SAVE_PATH + 'freq_hires_%s.npy' % ts15, freq_hires)
    np.save(SAVE_PATH + 'raw_hires_%s.npy'  % ts15, raw_hr)

    # Plot RHINO band
    f_rhino_h = freq_hires[mask_rhino_h]
    s_rhino_h = spec_hr[mask_rhino_h]
    fig, ax   = plt.subplots(figsize=(11, 4))
    ax2 = ax.twiny()
    ax.plot(f_rhino_h, s_rhino_h, lw=0.5, color='steelblue', alpha=0.85,
            label='Hi-res FFT (%.3f kHz/bin)' % DF_HIRES_KHZ)
    ax.set_xlabel('Frequency (MHz)')
    ax.set_ylabel('Power (dB, arb.)')
    ax.set_xlim(RHINO_LO_MHZ, RHINO_HI_MHZ)
    ax.set_title('Hi-res Spectrum %.3f kHz/bin | N=%d | %s' %
                 (DF_HIRES_KHZ, N_FFT_HIRES, ANTENNA_LABEL))
    ax.legend(fontsize=8)
    f_ticks = np.linspace(RHINO_LO_MHZ, RHINO_HI_MHZ, 6)
    ax2.set_xlim(RHINO_LO_MHZ, RHINO_HI_MHZ)
    ax2.set_xticks(f_ticks)
    ax2.set_xticklabels(['z=%.1f' % (1420.406/f - 1) for f in f_ticks], fontsize=8)
    ax2.set_xlabel('Redshift z', fontsize=9)
    fig.tight_layout()
    out15 = SAVE_PATH + 'thesis_fig6_hires_%s.png' % ts15
    fig.savefig(out15, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print('Saved:', out15)
    print('PASS Cell 15 — hi-res spectrum complete')

Capturing hi-res spectrum...
Saved: /home/xilinx/jupyter_notebooks/RHINO/data/Yagi_LNA/thesis_fig6_hires_20260520_002851.png
PASS Cell 15 — hi-res spectrum complete


In [32]:
# ================================================================
# CELL 16 — Hi-res waterfall
# Captures multiple hi-res spectra and builds a time-frequency
# waterfall at 4.219 kHz/bin. Shows persistent narrowband RFI
# as horizontal stripes and transient events as bright spots.
# ================================================================
if not atten_ok:
    print('SKIP — fix clipping first')
else:
    ts16    = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
    N_HR_FR = 40   # number of hi-res frames
    NT_HR   = NT_HIRES
    n_bins_hr = rhino_hi_h - rhino_lo_h

    wf_hires = np.zeros((N_HR_FR, n_bins_hr), dtype=np.float32)
    hr_times = np.zeros(N_HR_FR)
    t_hr     = time.time()

    print('Capturing %d hi-res frames...' % N_HR_FR)
    for fr in range(N_HR_FR):
        t_fr_start = time.time()
        raw = _ddr4_capture_raw(NT_HR)
        spec = fft_hires(raw)
        wf_hires[fr] = spec[rhino_lo_h:rhino_hi_h]
        hr_times[fr] = time.time() - t_hr
        if (fr + 1) % 10 == 0:
            print('  Hi-res frame %d/%d | t=%.1fs' % (fr+1, N_HR_FR, hr_times[fr]))
        elapsed_fr = time.time() - t_fr_start
        if elapsed_fr < WF_FRAME_TARGET_S:
            time.sleep(WF_FRAME_TARGET_S - elapsed_fr)

    np.save(SAVE_PATH + 'wf_hires_%s.npy' % ts16, wf_hires)

    f_hr = np.linspace(RHINO_LO_MHZ, RHINO_HI_MHZ, n_bins_hr)
    # Use percentile-based colour scale so Phil can see the dynamic range
    flat = wf_hires[wf_hires != 0].ravel()
    vmin_hr = float(np.percentile(flat, 5))  if len(flat) else 30
    vmax_hr = float(np.percentile(flat, 95)) if len(flat) else 60

    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    im = axes[0].imshow(wf_hires, aspect='auto', origin='upper',
                        extent=[f_hr[0], f_hr[-1], hr_times[-1], 0],
                        cmap='viridis', vmin=vmin_hr, vmax=vmax_hr)
    axes[0].set_xlabel('Frequency (MHz)')
    axes[0].set_ylabel('Time (s)')
    axes[0].set_title('Hi-res FFT Waterfall | %.3f kHz/bin | %d frames' %
                      (DF_HIRES_KHZ, N_HR_FR))
    plt.colorbar(im, ax=axes[0], label='dB (%.0f–%.0f dB range)' % (vmin_hr, vmax_hr))

    axes[1].plot(f_hr, np.mean(wf_hires, axis=0),
                 color='steelblue', lw=0.7, alpha=0.8, label='Mean')
    axes[1].plot(f_hr, np.max(wf_hires, axis=0),
                 color='red', lw=0.7, alpha=0.6, label='Max-hold')
    axes[1].set_xlabel('Frequency (MHz)')
    axes[1].set_ylabel('Power (dB, arb.)')
    axes[1].set_title('Mean and Max-hold | %d frames' % N_HR_FR)
    axes[1].set_xlim(RHINO_LO_MHZ, RHINO_HI_MHZ)
    axes[1].legend(fontsize=9)

    fig.suptitle('Hi-res Waterfall | RHINO 60–85 MHz | %s' % ANTENNA_LABEL)
    fig.tight_layout()
    out16 = SAVE_PATH + 'thesis_fig7_hires_waterfall_%s.png' % ts16
    fig.savefig(out16, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print('Colour scale: %.1f – %.1f dB (5th–95th percentile)' % (vmin_hr, vmax_hr))
    print('Saved:', out16)
    print('PASS Cell 16 — hi-res waterfall complete')

Capturing 40 hi-res frames...
  Hi-res frame 10/40 | t=90.6s
  Hi-res frame 20/40 | t=190.7s
  Hi-res frame 30/40 | t=290.8s
  Hi-res frame 40/40 | t=390.9s
Colour scale: 44.3 – 64.9 dB (5th–95th percentile)
Saved: /home/xilinx/jupyter_notebooks/RHINO/data/Yagi_LNA/thesis_fig7_hires_waterfall_20260520_002900.png
PASS Cell 16 — hi-res waterfall complete


In [28]:
# ================================================================
# CELL 17 — Integration / radiometer equation test
# Captures many hi-res spectra and averages them progressively.
# Checks that the noise decreases as 1/sqrt(N) as expected
# from the radiometer equation. Saves snapshots at regular
# intervals so the noise reduction can be plotted afterwards.
# ================================================================
if not atten_ok:
    print('SKIP — fix clipping first')
else:
    ts17  = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
    N_INT = 19800   # total number of spectra to integrate
    NT_INT = NT_HIRES
    SNAP_INTERVAL = 100   # save a snapshot every N spectra

    integ_sum  = np.zeros(N_FFT_HIRES // 2 + 1, dtype=np.float64)
    snap_Ns    = []
    t_int      = time.time()

    print('Starting integration: %d spectra...' % N_INT)
    for n in range(1, N_INT + 1):
        raw  = _ddr4_capture_raw(NT_INT)
        spec = fft_hires(raw).astype(np.float64)
        integ_sum += spec
        mean_spec  = (integ_sum / n).astype(np.float32)

        if n % SNAP_INTERVAL == 0 or n == N_INT:
            np.save(SAVE_PATH + 'integ_snap_%s_N%d.npy' % (ts17, n), mean_spec)
            snap_Ns.append(n)
            if n % 60 == 0:
                print('  N=%d | t=%.0fs' % (n, time.time()-t_int))

    np.save(SAVE_PATH + 'integ_final_%s.npy' % ts17, (integ_sum/N_INT).astype(np.float32))
    np.save(SAVE_PATH + 'integ_freq_%s.npy'  % ts17, freq_hires)
    print('Integration complete | %d snapshots saved' % len(snap_Ns))
    print('PASS Cell 17 — integration complete')

Starting integration: 19800 spectra...
  N=300 | t=161s
  N=600 | t=322s
  N=900 | t=483s
  N=1200 | t=643s
  N=1500 | t=804s
  N=1800 | t=965s
  N=2100 | t=1125s
  N=2400 | t=1286s
  N=2700 | t=1448s
  N=3000 | t=1609s
  N=3300 | t=1769s
  N=3600 | t=1930s
  N=3900 | t=2091s
  N=4200 | t=2252s
  N=4500 | t=2413s
  N=4800 | t=2573s
  N=5100 | t=2734s
  N=5400 | t=2894s
  N=5700 | t=3055s
  N=6000 | t=3216s
  N=6300 | t=3377s
  N=6600 | t=3538s
  N=6900 | t=3699s
  N=7200 | t=3860s
  N=7500 | t=4021s
  N=7800 | t=4183s
  N=8100 | t=4343s
  N=8400 | t=4504s
  N=8700 | t=4665s
  N=9000 | t=4826s
  N=9300 | t=4987s
  N=9600 | t=5148s
  N=9900 | t=5309s
  N=10200 | t=5469s
  N=10500 | t=5630s
  N=10800 | t=5791s
  N=11100 | t=5952s
  N=11400 | t=6114s
  N=11700 | t=6274s
  N=12000 | t=6435s
  N=12300 | t=6596s
  N=12600 | t=6757s
  N=12900 | t=6918s
  N=13200 | t=7079s
  N=13500 | t=7239s
  N=13800 | t=7401s
  N=14100 | t=7561s
  N=14400 | t=7722s
  N=14700 | t=7883s
  N=15000 | t=8045s
  N

In [34]:
# ================================================================
# CELL 18 — Radiometer plot
# Loads the integration snapshots saved in Cell 17 and plots
# the noise (spectral std dev) vs number of averaged spectra N.
# Compares against the ideal 1/sqrt(N) radiometer equation.
# ================================================================
if not atten_ok:
    print('SKIP — fix clipping first')
else:
    import glob
    snap_files = sorted(glob.glob(SAVE_PATH + 'integ_snap_%s_N*.npy' % ts17))
    ref_mask   = (freq_hires >= 60.0) & (freq_hires <= 75.0)   # quiet reference band

    Ns   = []
    stds = []
    for fp in snap_files:
        arr = np.load(fp, allow_pickle=False)
        if arr.size == len(freq_hires) and ref_mask.sum() > 0:
            N_val = int(fp.split('_N')[-1].replace('.npy', ''))
            std   = float(np.std(arr[ref_mask]))
            if np.isfinite(std) and std > 0:
                Ns.append(N_val)
                stds.append(std)

    if len(Ns) > 2:
        Ns   = np.array(Ns,   dtype=float)
        stds = np.array(stds, dtype=float)
        scale = stds[0] * np.sqrt(Ns[0])
        N_id  = np.geomspace(Ns.min(), Ns.max(), 100)
        ratio = (stds[0]*np.sqrt(Ns[0])) / (stds[-1]*np.sqrt(Ns[-1]))

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(N_id, scale/np.sqrt(N_id), 'k--', lw=1.5,
                alpha=0.5, label='Ideal 1/√N')
        ax.scatter(Ns, stds, s=20, color='steelblue', zorder=5)
        ax.plot(Ns, stds, color='steelblue', lw=1.0,
                label='Measured (ratio=%.2f)' % ratio)
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('N (averaged spectra)')
        ax.set_ylabel('Std dev in 60–75 MHz (dB)')
        ax.set_title('Radiometer Equation | %s | compliance ratio=%.2f' %
                     (ANTENNA_LABEL, ratio))
        ax.legend(fontsize=9)
        fig.tight_layout()
        out18 = SAVE_PATH + 'thesis_fig8_sensitivity_%s.png' % ts17
        fig.savefig(out18, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print('Compliance ratio: %.2f' % ratio)
        print('Saved:', out18)
    print('PASS Cell 18 — radiometer plot complete')

Compliance ratio: 1.04
Saved: /home/xilinx/jupyter_notebooks/RHINO/data/Yagi_LNA/thesis_fig8_sensitivity_20260520_003622.png
PASS Cell 18 — radiometer plot complete


In [29]:
# ================================================================
# CELL 19 — Session summary
# Prints a complete summary of everything captured in this session
# and lists all saved files with their shapes.
# ================================================================
import glob as _glob
print('=' * 60)
print('SESSION SUMMARY')
print('=' * 60)
print('Mode      :', OBSERVATION_MODE)
print('Antenna   :', ANTENNA_LABEL)
print('N_TAPS    :', N_TAPS)
print('Band      : %.0f – %.0f MHz' % (RHINO_LO_MHZ, RHINO_HI_MHZ))
print()
print('Saved files:')
for fp in sorted(_glob.glob(SAVE_PATH + '*.npy') + _glob.glob(SAVE_PATH + '*.png')):
    try:
        if fp.endswith('.npy'):
            arr = np.load(fp, allow_pickle=False)
            print('  %-60s shape=%s' % (os.path.basename(fp), str(arr.shape)))
        else:
            size_kb = os.path.getsize(fp) / 1024
            print('  %-60s %.0f KB' % (os.path.basename(fp), size_kb))
    except Exception:
        print('  %-60s (unreadable)' % os.path.basename(fp))
print()
print('PASS Cell 19 — session complete')

SESSION SUMMARY
Mode      : jbo
Antenna   : 50Ohm Load + ZKL-2+ LNA (JBO)
N_TAPS    : 4
Band      : 60 – 85 MHz

Saved files:
  fft_coarse_20260520_002628.npy                               shape=(8193,)
  freq_coarse_20260520_002628.npy                              shape=(8193,)
  integ_final_20260520_003037.npy                              shape=(524289,)
  integ_freq_20260520_003037.npy                               shape=(524289,)
  integ_snap_20260520_002648_N108.npy                          shape=(524289,)
  integ_snap_20260520_002648_N12.npy                           shape=(524289,)
  integ_snap_20260520_002648_N120.npy                          shape=(524289,)
  integ_snap_20260520_002648_N132.npy                          shape=(524289,)
  integ_snap_20260520_002648_N144.npy                          shape=(524289,)
  integ_snap_20260520_002648_N156.npy                          shape=(524289,)
  integ_snap_20260520_002648_N168.npy                          shape=(524289,)
  integ_s